# MSc in AI Capstone Project 4:Deep Learning Systems 

## Landmark Image Classification

**Task type:** Image classification using a Convolutional Neural Network (CNN)

This notebook implements a baseline CNN trained from scratch, an experimental variant that isolates a single change to the training configuration, and a direct comparison of the two on a held-out test set.

## Environment Checks

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device


---
## Task 1 — Problem Definition and Dataset Selection

**Task type:** Image-based classification using a Convolutional Neural Network (CNN).

**Problem:** given a user-supplied photo, predict which of 50 landmarks it depicts. This is a real problem faced by photo-sharing and photo-storage services, which want location metadata for uploaded photos but often don't have it (missing GPS data, scrubbed metadata, etc.). A landmark classifier can infer likely location directly from image content.

**Dataset:** a curated 50-class, ~6,000-image subset of Google's [Landmarks Dataset v2](https://github.com/cvdfoundation/google-landmark) (which itself contains 5M+ images across 200K+ landmarks). This subset was prepared for coursework use and distributed via Udacity's Deep Learning Nanodegree at:

`https://udacity-dlnfd.s3-us-west-1.amazonaws.com/datasets/landmark_images.zip`

**Dataset requirements checklist:**
- **Appropriate for the task** — real-world photographs of 50 distinct landmark categories, directly suited to CNN-based image classification.
- **Sufficient samples** — ~4,996 training/validation images and 1,250 held-out test images, roughly 100–140 images per class.
- **Publicly available** — downloadable via the URL above; the code cell below (`setup_env()`) downloads and extracts it automatically.
- **Not synthetic/AI-generated** — real photographs sourced from Wikimedia Commons and crowd-sourced landmark photography as part of Google's Landmarks v2 collection effort.
- **Not reused from a prior capstone** — this dataset was not used in any previous capstone submission in this MSc program.

**Access and preparation:** `src/helpers.py::setup_env()` downloads and unzips the dataset into a local `landmark_images/` directory (pre-split into `train/` and `test/` subfolders, each with 50 class subfolders), then computes and caches per-channel mean/std for normalization. `src/data.py::get_data_loaders()` wraps this into `torchvision.datasets.ImageFolder`-based train/validation/test `DataLoader`s with resizing (256), cropping (224), light augmentation (random crop + horizontal flip on the training split only), and normalization.

In [ ]:
from src.helpers import setup_env

# Downloads and extracts the dataset (skipped if already present),
# and computes/caches the per-channel normalization statistics.
setup_env()


---
## Task 2 — Load and Inspect the Dataset

In [ ]:
from src.data import get_data_loaders, visualize_one_batch

data_loaders = get_data_loaders(batch_size=16, valid_size=0.1, num_workers=2)

for split, dl in data_loaders.items():
    print(f"{split}: {len(dl.dataset)} images, {len(dl.dataset.classes)} classes")


### Representative samples

In [ ]:
%matplotlib inline
visualize_one_batch(data_loaders, max_n=5)


### Input shapes, labels, and data types

In [ ]:
images, labels = next(iter(data_loaders["train"]))
print("Batch image tensor shape:", images.shape)   # [batch, channels, height, width]
print("Image dtype:", images.dtype)
print("Batch label tensor shape:", labels.shape)
print("Label dtype:", labels.dtype)
print("Number of classes:", len(data_loaders["train"].dataset.classes))
print("Example class folder names (label = folder index):", data_loaders["train"].dataset.classes[:5])


**Data quality / preprocessing notes:**
- Images arrive at varied native resolutions and aspect ratios; the pipeline resizes to 256px on the short side then center/random-crops to a uniform 224×224 before batching.
- Class labels are inferred from the 50 folder names (`ImageFolder` convention) rather than a separate label file — folder names are numeric landmark IDs, not human-readable landmark names, which is a minor interpretability limitation revisited in the Limitations section of the report.
- The training split applies `RandomResizedCrop` and `RandomHorizontalFlip` as augmentation; validation and test splits use a deterministic `Resize` + `CenterCrop` so evaluation is not affected by randomness.
- Per-channel mean/std normalization is computed directly from the training data (cached to `mean_and_std.pt`) rather than reusing generic ImageNet statistics, since this is a different, smaller, and more visually narrow (landmark-photo) distribution.

---
## Task 3 — Baseline Deep Learning Model

`src/model.py` defines `MyModel`, a CNN built from scratch (no transfer learning). It consists of 5 convolutional blocks (`Conv2d → BatchNorm2d → ReLU → MaxPool2d`), progressively increasing channel depth from 32 to 512 while halving spatial resolution at each block (224 → 7), followed by a fully-connected classifier head (`Flatten → Linear(25088, 1024) → ReLU → Dropout → Linear(1024, num_classes)`).

**Key design choices:**
- **BatchNorm after every conv layer** stabilizes training and tolerates a higher learning rate than an unnormalized network.
- **MaxPool2d(2,2) after every block** keeps the parameter count manageable while progressively growing the receptive field.
- **Dropout before the final linear layer** regularizes the large (25,088 → 1,024) fully-connected transition, the part of the network most prone to overfitting given the modest dataset size.
- **Raw logit output** (no final softmax) — `nn.CrossEntropyLoss` applies `LogSoftmax` internally, so the model itself outputs unnormalized class scores.
- **Loss and optimizer:** cross-entropy loss (`src/optimization.py::get_loss`), appropriate for single-label multi-class classification; SGD with momentum (`src/optimization.py::get_optimizer`) as the baseline optimizer, a standard, well-understood starting point for a from-scratch CNN.

In [ ]:
from src.model import MyModel

baseline_model = MyModel(num_classes=50, dropout=0.3)
baseline_model


### Baseline training configuration

The baseline uses SGD with momentum — a conservative, well-understood choice for training a CNN from scratch.

In [ ]:
# ---- Baseline hyperparameters ----
batch_size = 16
valid_size = 0.1
num_epochs = 15          # kept modest for CPU feasibility; increase if a GPU is available
num_classes = 50          # do not change — fixed by the dataset
dropout = 0.3
learning_rate = 0.01
baseline_optimizer_name = "sgd"
weight_decay = 0.0001

data_loaders = get_data_loaders(batch_size=batch_size, valid_size=valid_size, num_workers=2)


In [ ]:
from src.train import optimize
from src.optimization import get_optimizer, get_loss

baseline_model = MyModel(num_classes=num_classes, dropout=dropout)

baseline_opt = get_optimizer(
    baseline_model,
    optimizer=baseline_optimizer_name,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
)

loss_fn = get_loss()

baseline_train_losses, baseline_valid_losses = optimize(
    data_loaders,
    baseline_model,
    baseline_opt,
    loss_fn,
    n_epochs=num_epochs,
    save_path="checkpoints/baseline_best_val_loss.pt",
    interactive_tracking=False,
)


### Baseline training progress

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(baseline_train_losses) + 1), baseline_train_losses, label="Train loss")
plt.plot(range(1, len(baseline_valid_losses) + 1), baseline_valid_losses, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Baseline model — training vs. validation loss")
plt.legend()
plt.show()


---
## Task 4 — Experimental Comparison

**What was changed:** the optimizer — SGD with momentum (baseline) → **Adam** (experimental) — with every other setting (architecture, learning rate, batch size, epochs, dropout, weight decay) held fixed.

**Why this change:** the optimizer governs how gradients update weights, directly shaping convergence speed and the quality of the minimum reached, independent of model capacity or data. SGD with momentum is simple and well-understood, but can converge slowly and is sensitive to learning-rate choice. Adam adds per-parameter adaptive learning rates (via running estimates of gradient mean and variance), which typically converges faster and is more forgiving of a fixed global learning rate — a meaningful, well-motivated axis to isolate and test on this dataset.

**How the experimental setup differs from the baseline:** identical `MyModel` architecture, identical data pipeline, identical hyperparameters (learning rate, batch size, epoch count, dropout, weight decay) — the *only* difference is `optimizer="adam"` instead of `optimizer="sgd"` in `get_optimizer()`.

In [ ]:
experimental_optimizer_name = "adam"

experimental_model = MyModel(num_classes=num_classes, dropout=dropout)

experimental_opt = get_optimizer(
    experimental_model,
    optimizer=experimental_optimizer_name,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
)

experimental_train_losses, experimental_valid_losses = optimize(
    data_loaders,
    experimental_model,
    experimental_opt,
    loss_fn,
    n_epochs=num_epochs,
    save_path="checkpoints/experimental_best_val_loss.pt",
    interactive_tracking=False,
)


### Experimental model training progress

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(experimental_train_losses) + 1), experimental_train_losses, label="Train loss")
plt.plot(range(1, len(experimental_valid_losses) + 1), experimental_valid_losses, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Experimental model (Adam) — training vs. validation loss")
plt.legend()
plt.show()


---
## Task 5 — Evaluate and Compare Model Performance

In [ ]:
from src.train import one_epoch_test

print("Baseline (SGD) — test set evaluation:")
baseline_model.load_state_dict(torch.load("checkpoints/baseline_best_val_loss.pt"))
baseline_test_loss, baseline_test_acc = one_epoch_test(data_loaders["test"], baseline_model, loss_fn)

print("\nExperimental (Adam) — test set evaluation:")
experimental_model.load_state_dict(torch.load("checkpoints/experimental_best_val_loss.pt"))
experimental_test_loss, experimental_test_acc = one_epoch_test(data_loaders["test"], experimental_model, loss_fn)


### Side-by-side loss curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

axes[0].plot(range(1, len(baseline_train_losses) + 1), baseline_train_losses, label="Train")
axes[0].plot(range(1, len(baseline_valid_losses) + 1), baseline_valid_losses, label="Validation")
axes[0].set_title("Baseline (SGD)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(range(1, len(experimental_train_losses) + 1), experimental_train_losses, label="Train")
axes[1].plot(range(1, len(experimental_valid_losses) + 1), experimental_valid_losses, label="Validation")
axes[1].set_title("Experimental (Adam)")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


### Quantitative comparison

In [ ]:
comparison = pd.DataFrame({
    "Configuration": ["Baseline (SGD)", "Experimental (Adam)"],
    "Final train loss": [baseline_train_losses[-1], experimental_train_losses[-1]],
    "Final valid loss": [baseline_valid_losses[-1], experimental_valid_losses[-1]],
    "Test loss": [float(baseline_test_loss), float(experimental_test_loss)],
    "Test accuracy (%)": [float(baseline_test_acc), float(experimental_test_acc)],
})
comparison


In [ ]:
plt.figure(figsize=(6, 5))
plt.bar(comparison["Configuration"], comparison["Test accuracy (%)"], color=["#4C72B0", "#DD8452"])
plt.ylabel("Test accuracy (%)")
plt.title("Baseline vs. Experimental — test accuracy")
plt.show()


**Trade-off summary:** compare the two configurations above on convergence speed (how quickly validation loss drops in the early epochs), final validation loss, and held-out test accuracy. A faster-converging but not-necessarily-more-accurate optimizer, or vice versa, is a common and instructive outcome — the actual numbers from the run above determine which pattern applies here, and that reading is carried into the Results and Interpretation section of the accompanying report.

---
## Task 6 — Notebook Summary

This notebook trained a from-scratch CNN (`MyModel`: 5 convolutional blocks with batch normalization, followed by a dropout-regularized fully-connected classifier) to classify images into one of 50 landmark categories, using a curated ~6,000-image subset of Google's Landmarks Dataset v2. Two configurations were compared: a baseline trained with SGD (momentum, weight decay) and an experimental variant identical in every respect except the optimizer, which was switched to Adam. Both models were trained for the same number of epochs on the same data splits, with a `ReduceLROnPlateau` scheduler and best-validation-loss checkpointing. The comparison above reports final train/validation loss curves, held-out test loss, and held-out test accuracy for both configurations, letting the effect of the optimizer choice be read off directly rather than inferred. As with any single-run comparison on a modest-sized dataset, results carry some run-to-run variance; the Limitations and Future Improvements sections of the accompanying report discuss this and other caveats (small per-class sample counts, non-descriptive numeric class labels, and the value of averaging over multiple seeds) in more depth.

---
## Task 8 — Reproducibility (`requirements.txt`)

Run the cell below **after** executing this notebook end-to-end in your working environment, so that `pip freeze` captures the exact package versions actually used.

In [ ]:
!pip freeze > requirements.txt
print("requirements.txt written.")
